In [1]:
import pandas as pd
import os

# all disease

In [2]:
from scipy import stats
import statsmodels.api as sm
import scipy.stats as st
import numpy as np

def all_diff(filtered_df, method1, method2, metric):
    list1 = filtered_df[filtered_df['method']==method1].sort_values(by='disease')[metric].tolist()
    list2 = filtered_df[filtered_df['method']==method2].sort_values(by='disease')[metric].tolist()

    t_stat, p_value = stats.ttest_rel(
        list1,
        list2,
        alternative="greater"   # list1 > list2
    )
    return p_value

def diff_category(filtered_df,method1, method2,metric):
    subdf = filtered_df[(filtered_df['method']==method1)|(filtered_df['method']==method2)][['method','disease',metric,'category']]
    # pivot to wide format
    wide = (
        subdf
        .pivot(index="disease", columns="method", values=metric)
        .dropna()
        .reset_index()
    )

    # name methods explicitly (adjust if needed)
    method1 = wide.columns[1]
    method2 = wide.columns[2]

    wide["diff"] = wide[method1] - wide[method2]
    cat_map = subdf[["disease", "category"]].drop_duplicates()
    wide = wide.merge(cat_map, on="disease")

    y = wide["diff"]
    X = pd.DataFrame({"const": 1.0}, index=wide.index)   # explicit intercept name

    ols = sm.OLS(y, X).fit(
        cov_type="cluster",
        cov_kwds={"groups": wide["category"]}
    )

    beta = ols.params["const"]
    se   = ols.bse["const"]

    z = beta / se
    p_one_sided = 1 - st.norm.cdf(z)   # H1: beta > 0

    prop_diseases = (wide["diff"] > 0).mean()

    cat_means = (wide.groupby("category", observed=True)["diff"].mean())

    prop_categories = (cat_means > 0).mean()

    return round(prop_diseases,2),round(prop_categories,2),beta,p_one_sided

def paired_cluster_test(df, m1, m2, metric):
    wide = (
        df[df["method"].isin([m1, m2])]
        .pivot_table(index="disease", columns="method", values=metric, aggfunc="mean")
        .dropna()
        .reset_index()
    )

    wide["diff"] = wide[m1] - wide[m2]

    cat_map = df[["disease", "category"]].drop_duplicates()
    wide = wide.merge(cat_map, on="disease")

    X = np.ones((len(wide), 1))
    ols = sm.OLS(wide["diff"], X).fit(
        cov_type="cluster",
        cov_kwds={"groups": wide["category"]}
    )

    beta = float(ols.params.iloc[0])
    se = float(ols.bse.iloc[0])

    if wide["diff"].nunique(dropna=True) <= 1:
        beta = float(wide["diff"].mean())  # will be 0
        se = 0.0
        p_two_sided = 1.0  # no difference
        return beta, se, p_two_sided

    z = beta / se
    p_one_sided = 1 - st.norm.cdf(z)

    return beta, se, p_one_sided

def paired_cluster_test_two_sided(df, m1, m2, metric):
    wide = (
        df[df["method"].isin([m1, m2])]
        .pivot_table(index="disease", columns="method", values=metric, aggfunc="mean")
    )

    # ensure both columns exist, then drop missing pairs
    if (m1 not in wide.columns) or (m2 not in wide.columns):
        return np.nan, np.nan, np.nan

    wide = wide.dropna(subset=[m1, m2]).reset_index()
    wide["diff"] = wide[m1] - wide[m2]

    # disease -> category (assumes one category per disease; keeps first if duplicated)
    cat_map = df[["disease", "category"]].drop_duplicates(subset=["disease"])
    wide = wide.merge(cat_map, on="disease", how="left")

    X = np.ones((len(wide), 1))
    ols = sm.OLS(wide["diff"], X).fit(
        cov_type="cluster",
        cov_kwds={"groups": wide["category"]}
    )

    beta = float(ols.params.iloc[0])
    se = float(ols.bse.iloc[0])

    if wide["diff"].nunique(dropna=True) <= 1:
        beta = float(wide["diff"].mean())  # will be 0
        se = 0.0
        p_two_sided = 1.0  # no difference
        return beta, se, p_two_sided

    z = beta / se
    p_two_sided = 2 * (1 - st.norm.cdf(abs(z)))

    return beta, se, p_two_sided

def paired_test_from_wide(wide: "pd.DataFrame", m1: str, m2: str):
    """
    wide: index = disease, columns = methods, values = metric
    Returns: beta (mean diff m1-m2), se, p(two-sided)
    """
    x = wide[m1].to_numpy(dtype=float)
    y = wide[m2].to_numpy(dtype=float)

    diff = x - y
    diff = diff[np.isfinite(diff)]
    n = diff.size

    if n < 2:
        return float(np.nanmean(diff)) if n == 1 else np.nan, np.nan, np.nan

    beta = float(diff.mean())
    sd = float(diff.std(ddof=1))
    se = sd / np.sqrt(n)

    # handle degenerate cases (your ZeroDivisionError)
    if se == 0.0 or not np.isfinite(se):
        p = 1.0 if np.isclose(beta, 0.0) else 0.0
        return beta, se, p

    t = beta / se
    p_two_sided = float(2 * st.t.sf(np.abs(t), df=n - 1))
    return beta, se, p_two_sided

In [14]:
statics = []

In [4]:
merged_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/all_metrics_1and2percent.csv')
merged_df = merged_df[['disease', 'setting', 'method', 'category', 'bio_sim_300',
       'bio_sim_150', 'top_recall_150', 'succ_150', 
       'top_recall_300', 'succ_300',  'bedroc_150',
       'bedroc_300', 'auroc']]

selected_metrics= ['bio_sim_300',
       'bio_sim_150', 'top_recall_150', 'succ_150', 
       'top_recall_300', 'succ_300', 'bedroc_150',
       'bedroc_300', 'auroc']

### OCC

In [5]:
import pandas as pd

selected_pairs = [('df_gnn_occsvm_pred', 'occ_svm'),
    ('df_gnn_occsvm_pred', 'ppi_emb_n2v'),
    ('2019_nn_all_pred', 'ppi_emb_n2v'),
    ('2019_occ_deep_svd_pred/pred.pkl', 'uniport_ppi_2019')]
subresults = (merged_df.set_index(['setting', 'method']).loc[selected_pairs].reset_index())

method_map = {
    ('df_gnn_occsvm_pred', 'occ_svm'): 'occ-svm',
    ('df_gnn_occsvm_pred', 'ppi_emb_n2v'): 'bagging-svm',
    ('2019_nn_all_pred', 'ppi_emb_n2v'): 'bagging-dnn',
    ('2019_occ_deep_svd_pred/pred.pkl', 'uniport_ppi_2019'): 'occ-deepsvdd',
}

# (optional but recommended) normalize strings to avoid hidden whitespace issues
subresults['setting'] = subresults['setting'].astype(str).str.strip()
subresults['method']  = subresults['method'].astype(str).str.strip()

# build a MultiIndex key per row and map using the dict
keys = pd.MultiIndex.from_frame(subresults[['setting', 'method']])
new_method = keys.map(method_map)  # returns array with mapped values or NaN

# apply mapped names where available
subresults.loc[~pd.isna(new_method), 'method'] = new_method[~pd.isna(new_method)]

# Compute mean AUROC per category
cat_order = (
    subresults.groupby("category")["auroc"]
    .mean()
    .sort_values(ascending=False)
)

# Sort the whole DataFrame by that category order
subresults["category"] = pd.Categorical(
    subresults["category"],
    categories=cat_order.index,
    ordered=True
)

subresults = subresults.sort_values("category").reset_index(drop=True)

In [6]:
for pair in [['bagging-svm', 'occ-svm'], ['bagging-dnn', 'occ-deepsvdd']]:
    for metric in selected_metrics:

        wide = (
            subresults[subresults["method"].isin(pair)]
            .pivot_table(index="disease", columns="method", values=metric, aggfunc="mean")
        )

        # skip if either method column missing
        if not set(pair).issubset(wide.columns):
            continue

        wide = wide.dropna(subset=pair)

        mean1 = wide[pair[0]].mean()
        mean2 = wide[pair[1]].mean()
        better, worse = (pair[0], pair[1]) if mean1 >= mean2 else (pair[1], pair[0])

        # beta, se, p_value = paired_cluster_test_two_sided(subresults, pair[0], pair[1], metric)
        beta, se, p_value = paired_cluster_test(subresults, pair[0], pair[1], metric)

        single_stat = ['occ_vs_bagging', metric, better, worse, beta, se, p_value]
        statics.append(single_stat)

### feature

In [7]:
merged_df['setting'].unique()

array(['df_gnn_occsvm_pred', '2019_occ_deep_svd_pred/pred.pkl',
       '2019_nn_all_pred', '2019_rf_renamed_pred', '2019_mf_add_pred',
       'graphsage', 'gcn'], dtype=object)

In [15]:
selected_features = ['ppi_emb_df','ppi_emb_dw', 'ppi_emb_n2v', 'literature_emb','seq_emb_esm', 'seq_emb_port']
# settings = ['df_gnn_occsvm_pred','2019_nn_all_pred','2019_rf_renamed_pred','2019_mf_add_pred','graphsage', 'gcn']
settings = ['2019_mf_add_pred','graphsage', 'gcn']

map_name = {
    'df_gnn_occsvm_pred':'SVM_features',
    '2019_nn_all_pred':'DNN_features',
    '2019_rf_renamed_pred':'RF_features',
    '2019_mf_add_pred': 'MF_features',
    'graphsage': 'GraphSAGE_features', 
    'gcn':'GCN_features'
}

for setting in settings:
    for metric in selected_metrics:
        filtered_df = merged_df[(merged_df['setting']==setting)&(merged_df['method'].isin(selected_features))]
        wide = (
            filtered_df.pivot_table(index="disease", columns="method", values=metric, aggfunc="mean")
            .dropna())

        mean_auroc = wide.mean().sort_values(ascending=False)
        best_method = mean_auroc.index[0]

        methods = wide.columns.tolist()
        others = [m for m in methods if m != best_method]

        results = []
        for m in others:
            # beta, se, p = paired_cluster_test_two_sided(filtered_df, best_method, m, metric)
            beta, se, p = paired_cluster_test(filtered_df, best_method, m, metric)

            single_stat = [map_name[setting], metric, best_method, m, beta, se, p]
            statics.append(single_stat)
    

### models

In [16]:
selected_pairs = [('df_gnn_occsvm_pred','ppi_emb_n2v'),
                ('graphsage','Graph_sage'),
                ('gcn','GCNs'),
            ('2019_nn_all_pred','ppi_emb_n2v'),
            ('2019_rf_renamed_pred','ppi_emb_n2v'),
            ('2019_mf_bag_2_pred','uniport_ppi_2019')]
for feature_i in selected_features:
    selected_pairs = [('df_gnn_occsvm_pred',feature_i),
                ('df_gnn_occsvm_pred',feature_i),
                ('df_gnn_occsvm_pred',feature_i),
            ('2019_nn_all_pred',feature_i),
            ('2019_rf_renamed_pred',feature_i),
            ('2019_mf_add_pred',feature_i)]
    subresults = (merged_df.set_index(['setting', 'method']).loc[selected_pairs].reset_index())
    method_mapping = dict(zip(selected_pairs,['SVM','GraphSAGE','GCN','DNN','RF','MF']))
    subresults['method'] = [method_mapping[(row['setting'], row['method'])]for _, row in subresults.iterrows()]
    for metric in selected_metrics:
        filtered_df = subresults.copy()
        wide = (
            filtered_df.pivot_table(index="disease", columns="method", values=metric, aggfunc="mean")
            .dropna())

        mean_auroc = wide.mean().sort_values(ascending=False)
        best_method = mean_auroc.index[0]

        methods = wide.columns.tolist()
        others = [m for m in methods if m != best_method]

        results = []
        for m in others:
            # beta, se, p = paired_cluster_test_two_sided(filtered_df, best_method, m, metric)
            beta, se, p = paired_cluster_test(filtered_df, best_method, m, metric)

            single_stat = ['Models_'+feature_i, metric, best_method, m, beta, se, p]
            statics.append(single_stat)

## fusion

In [10]:
# svm_features = ['ppi_emb_dw','early_fused', 'late_fused','mid_linear_fused','mid_geo_fused']
# svm_ppi = ['ppi_emb_dw','early_fused_ppi', 'late_fused_ppi', 'mid_linear_fused_ppi', 'mid_geo_fused_ppi']
# dnn_features = ['ppi_emb_df', 'early_fused', 'late_fused', 'mid_fused']
# dnn_ppi = ['ppi_emb_df','early_fused_ppi', 'late_fused_ppi','mid_fused_ppi']

svm_features = ['early_fused', 'late_fused','mid_linear_fused','mid_geo_fused']
svm_ppi = ['early_fused_ppi', 'late_fused_ppi', 'mid_linear_fused_ppi', 'mid_geo_fused_ppi']
dnn_features = ['early_fused', 'late_fused', 'mid_fused']
dnn_ppi = ['early_fused_ppi', 'late_fused_ppi','mid_fused_ppi']

settings_dict = {'Kernel_fused':['df_gnn_occsvm_pred',svm_features],
            'Kernel_fused_ppi':['df_gnn_occsvm_pred',svm_ppi],
            'DNN_fused':['2019_nn_all_pred',dnn_features],
            'DNN_fused_ppi':['2019_nn_all_pred',dnn_ppi]}

def debug_pair(sub, candidate_cols=("cluster", "fold", "seed", "run", "split", "dataset")):
    print("n rows:", len(sub))
    print("columns:", list(sub.columns))
    for c in candidate_cols:
        if c in sub.columns:
            print(f"n unique {c}:", sub[c].nunique())

for setting in list(settings_dict.keys()):
    for metric in selected_metrics:
        filtered_df = merged_df[(merged_df['setting']==settings_dict[setting][0])&(merged_df['method'].isin(settings_dict[setting][1]))]
        wide = (
            filtered_df.pivot_table(index="disease", columns="method", values=metric, aggfunc="mean")
            .dropna())

        mean_auroc = wide.mean().sort_values(ascending=False)
        best_method = mean_auroc.index[0]

        others = [m for m in wide.columns if m != best_method]

        for m in others:
            
            beta, se, p = paired_cluster_test(filtered_df, best_method, m, metric)

            # beta, se, p = paired_cluster_test_two_sided(filtered_df, best_method, m, metric)
            # beta, se, p = paired_test_from_wide(wide, best_method, m)

            single_stat = [setting, metric, best_method, m, beta, se, p]
            statics.append(single_stat)
    

## overall summary

In [17]:
statics_df = pd.DataFrame(
    statics,
    columns=['setting','metric','method1','method2','beta','se','p_two_sided']
)

# statics_df.to_csv(
#     '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/all_compara.csv',
    
# )


In [18]:
statics_df['setting'].unique()

array(['MF_features', 'GraphSAGE_features', 'GCN_features',
       'Models_ppi_emb_df', 'Models_ppi_emb_dw', 'Models_ppi_emb_n2v',
       'Models_literature_emb', 'Models_seq_emb_esm',
       'Models_seq_emb_port'], dtype=object)

In [19]:
statics_df[statics_df['p_two_sided'] < 0.005].round(4)

,setting,metric,method1,method2,beta,se,p_two_sided
0,MF_features,bio_sim_300,literature_emb,ppi_emb_df,0.0324,0.0096,0.0004
6,MF_features,bio_sim_150,ppi_emb_n2v,ppi_emb_df,0.0233,0.0077,0.0013
8,MF_features,bio_sim_150,ppi_emb_n2v,seq_emb_esm,0.0189,0.0059,0.0007
41,MF_features,auroc,ppi_emb_dw,ppi_emb_df,0.0541,0.0108,0.0000
46,GraphSAGE_features,bio_sim_300,ppi_emb_n2v,ppi_emb_df,0.0155,0.0044,0.0002
...,...,...,...,...,...,...,...
285,Models_seq_emb_port,succ_300,MF,DNN,0.2500,0.0924,0.0034
287,Models_seq_emb_port,succ_300,MF,RF,0.1667,0.0589,0.0023
294,Models_seq_emb_port,auroc,MF,DNN,0.1126,0.0177,0.0000
295,Models_seq_emb_port,auroc,MF,GCN,0.1046,0.0213,0.0000


In [12]:
statics_df[(statics_df['setting']=='Kernel_fused_ppi')&(statics_df['p_two_sided'] < 0.005)].round(4)

,setting,metric,method1,method2,beta,se,p_two_sided


In [13]:
# statics_df[(statics_df['setting']=='occ_vs_bagging')&(statics_df['p_two_sided'] < 0.005)].round(4)
# statics_df[(statics_df['setting']=='SVM_features')&(statics_df['p_two_sided'] < 0.005)].round(4)
# statics_df[(statics_df['setting']=='Models')&(statics_df['p_two_sided'] < 0.005)].round(4)
# statics_df[(statics_df['setting']=='Kernel_fused')&(statics_df['p_two_sided'] < 0.005)].round(4)
statics_df[(statics_df['setting']=='Kernel_fused_ppi')&(statics_df['p_two_sided'] < 0.05)].round(4)


,setting,metric,method1,method2,beta,se,p_two_sided
226,Kernel_fused_ppi,bio_sim_300,early_fused_ppi,mid_geo_fused_ppi,0.0130,0.0040,0.0005
227,Kernel_fused_ppi,bio_sim_300,early_fused_ppi,mid_linear_fused_ppi,0.0039,0.0022,0.0373
228,Kernel_fused_ppi,bio_sim_150,early_fused_ppi,late_fused_ppi,0.0051,0.0028,0.0379
229,Kernel_fused_ppi,bio_sim_150,early_fused_ppi,mid_geo_fused_ppi,0.0162,0.0060,0.0035
230,Kernel_fused_ppi,bio_sim_150,early_fused_ppi,mid_linear_fused_ppi,0.0065,0.0030,0.0137
235,Kernel_fused_ppi,succ_150,mid_linear_fused_ppi,late_fused_ppi,0.0625,0.0320,0.0255
237,Kernel_fused_ppi,top_recall_300,mid_geo_fused_ppi,early_fused_ppi,0.0678,0.0404,0.0468
240,Kernel_fused_ppi,succ_300,late_fused_ppi,early_fused_ppi,0.0625,0.0372,0.0465
249,Kernel_fused_ppi,auroc,mid_geo_fused_ppi,early_fused_ppi,0.0070,0.0036,0.0251
251,Kernel_fused_ppi,auroc,mid_geo_fused_ppi,mid_linear_fused_ppi,0.0047,0.0013,0.0002


In [14]:
# statics_df[(statics_df['setting']=='DNN_fused')&(statics_df['p_two_sided'] < 0.005)].round(4)
statics_df[(statics_df['setting']=='DNN_fused_ppi')&(statics_df['p_two_sided'] < 0.05)].round(4)
# statics_df[(statics_df['setting']=='DNN_features')&(statics_df['p_two_sided'] < 0.005)].round(4)


,setting,metric,method1,method2,beta,se,p_two_sided
271,DNN_fused_ppi,bio_sim_300,late_fused_ppi,mid_fused_ppi,0.0329,0.0094,0.0002
273,DNN_fused_ppi,bio_sim_150,late_fused_ppi,mid_fused_ppi,0.0248,0.0103,0.0078
282,DNN_fused_ppi,bedroc_150,mid_fused_ppi,early_fused_ppi,0.0272,0.0163,0.0472
283,DNN_fused_ppi,bedroc_150,mid_fused_ppi,late_fused_ppi,0.0435,0.0141,0.0010
285,DNN_fused_ppi,bedroc_300,mid_fused_ppi,late_fused_ppi,0.0425,0.0156,0.0032


In [15]:
statics_df[(statics_df['setting']=='RF_features')&(statics_df['p_two_sided'] < 0.005)].round(4)


,setting,metric,method1,method2,beta,se,p_two_sided
111,RF_features,bio_sim_300,literature_emb,seq_emb_esm,0.0369,0.0092,0.0000
112,RF_features,bio_sim_300,literature_emb,seq_emb_port,0.0319,0.0079,0.0000
115,RF_features,bio_sim_150,ppi_emb_dw,ppi_emb_n2v,0.0087,0.0020,0.0000
116,RF_features,bio_sim_150,ppi_emb_dw,seq_emb_esm,0.0374,0.0116,0.0006
117,RF_features,bio_sim_150,ppi_emb_dw,seq_emb_port,0.0310,0.0116,0.0037
121,RF_features,top_recall_150,ppi_emb_dw,seq_emb_esm,0.0927,0.0360,0.0050
131,RF_features,top_recall_300,ppi_emb_n2v,seq_emb_esm,0.1413,0.0378,0.0001
132,RF_features,top_recall_300,ppi_emb_n2v,seq_emb_port,0.1435,0.0477,0.0013
141,RF_features,bedroc_150,ppi_emb_n2v,seq_emb_esm,0.0800,0.0289,0.0028
146,RF_features,bedroc_300,ppi_emb_n2v,seq_emb_esm,0.1072,0.0316,0.0004


In [20]:
statics_df_rounded = statics_df.round(4)

statics_df_rounded[statics_df_rounded['p_two_sided'] < 0.005][['setting', 'method1', 'method2', 'metric', 'beta', 'se']]

,setting,method1,method2,metric,beta,se
0,MF_features,literature_emb,ppi_emb_df,bio_sim_300,0.0324,0.0096
6,MF_features,ppi_emb_n2v,ppi_emb_df,bio_sim_150,0.0233,0.0077
8,MF_features,ppi_emb_n2v,seq_emb_esm,bio_sim_150,0.0189,0.0059
41,MF_features,ppi_emb_dw,ppi_emb_df,auroc,0.0541,0.0108
46,GraphSAGE_features,ppi_emb_n2v,ppi_emb_df,bio_sim_300,0.0155,0.0044
...,...,...,...,...,...,...
285,Models_seq_emb_port,MF,DNN,succ_300,0.2500,0.0924
287,Models_seq_emb_port,MF,RF,succ_300,0.1667,0.0589
294,Models_seq_emb_port,MF,DNN,auroc,0.1126,0.0177
295,Models_seq_emb_port,MF,GCN,auroc,0.1046,0.0213


In [22]:
ori_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/all_compara_early_0.005.csv')

In [24]:
ori_df['setting'].unique()

array(['occ_vs_bagging', 'SVM_features', 'DNN_features', 'RF_features',
       'Models', 'Kernel_fused', 'Kernel_fused_ppi', 'DNN_fused',
       'DNN_fused_ppi'], dtype=object)

In [25]:
ori_df = ori_df[ori_df['setting'].isin(['occ_vs_bagging', 'SVM_features', 'DNN_features', 'RF_features',
 'Kernel_fused', 'Kernel_fused_ppi', 'DNN_fused',
       'DNN_fused_ppi'])]

In [26]:
df2 = statics_df_rounded[statics_df_rounded['p_two_sided'] < 0.005][['setting', 'method1', 'method2', 'metric', 'beta', 'se']]
df_stacked = pd.concat([ori_df, df2])

In [28]:
df_stacked['setting'].unique()

array(['occ_vs_bagging', 'SVM_features', 'DNN_features', 'RF_features',
       'Kernel_fused', 'Kernel_fused_ppi', 'DNN_fused', 'DNN_fused_ppi',
       'MF_features', 'GraphSAGE_features', 'GCN_features',
       'Models_ppi_emb_df', 'Models_ppi_emb_dw', 'Models_ppi_emb_n2v',
       'Models_literature_emb', 'Models_seq_emb_esm',
       'Models_seq_emb_port'], dtype=object)

In [29]:
df_stacked.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/all_0.005.csv',index=False)

In [31]:
df_stacked[df_stacked['setting']=='DNN_fused_ppi']

,setting,method1,method2,metric,beta,se
102,DNN_fused_ppi,late_fused_ppi,mid_fused_ppi,bio_sim_300,0.0329,0.0094
103,DNN_fused_ppi,mid_fused_ppi,late_fused_ppi,bedroc_150,0.0435,0.0141
104,DNN_fused_ppi,mid_fused_ppi,late_fused_ppi,bedroc_300,0.0425,0.0156


In [21]:
# # Round numeric columns to 4 decimals
# statics_df_rounded = statics_df.copy()
# numeric_cols = statics_df_rounded.select_dtypes(include='number').columns
# statics_df_rounded[numeric_cols] = statics_df_rounded[numeric_cols].round(4)

statics_df_rounded = statics_df.round(4)
# Save p < 0.005
statics_df_rounded[statics_df_rounded['p_two_sided'] < 0.005][['setting', 'method1', 'method2', 'metric', 'beta', 'se']].to_csv(
    '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/part_temp_0.005.csv',
    index=False
)

# # Save p < 0.05
# statics_df_rounded[(statics_df_rounded['p_two_sided'] < 0.05)&(statics_df_rounded['p_two_sided'] > 0.005)][['setting', 'method1', 'method2', 'metric', 'beta', 'se']].to_csv(
#     '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/all_compara_early_0.05.csv',
#     index=False
# )
